# Phase 01: Raw Data Audit and Modality Inventory

This notebook audits `vrdataset/dataPackage` before feature extraction. It treats `vrdataset` as read-only and saves all generated outputs under `experiments/phase_01_raw_data_modality_audit`.

This phase does not perform feature extraction, does not create the final modeling dataset, and does not train ML or HDC models.

In [1]:
from pathlib import Path
import sys

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [START_DIR, *START_DIR.parents]:
    if (candidate / "vrdataset" / "dataPackage").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not locate project root from {START_DIR}")

PHASE_DIR = PROJECT_ROOT / "experiments" / "phase_01_raw_data_modality_audit"
DATA_ROOT = PROJECT_ROOT / "vrdataset" / "dataPackage"

sys.dont_write_bytecode = True
sys.path.insert(0, str(PHASE_DIR / "scripts"))

print(f"Execution start directory: {START_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Read-only source data root: {DATA_ROOT}")
print(f"Output root: {PHASE_DIR}")
print(f"Source exists: {DATA_ROOT.exists()}")

Execution start directory: E:\hdc-vr-pilot\experiments\phase_01_raw_data_modality_audit\notebooks
Project root: E:\hdc-vr-pilot
Read-only source data root: E:\hdc-vr-pilot\vrdataset\dataPackage
Output root: E:\hdc-vr-pilot\experiments\phase_01_raw_data_modality_audit
Source exists: True


## Execute the Audit

The audit recursively scans the requested extensions, parses subject/session/difficulty/run identifiers, detects modality using filename keywords and safe CSV/MAT metadata inspection, checks readability, and writes all requested Phase 01 outputs.

In [2]:
from IPython.display import display
from phase01_audit import run_audit

outputs = run_audit(PROJECT_ROOT)
inventory = outputs["inventory"]
run_availability = outputs["run_availability"]
summaries = outputs["summaries"]
report = outputs["report"]

print("Audit completed.")
print(f"Total files: {report['total_files']}")
print(f"Readable files: {report['total_readable_files']}")
print(f"Subjects: {report['total_subjects']}")
print(f"Sessions: {report['total_sessions']}")
print(f"Runs: {report['total_runs']}")

Audit completed.
Total files: 9003
Readable files: 9003
Subjects: 35
Sessions: 35
Runs: 487


## File-Level Inventory Preview

The full file-level inventory is saved to `results/raw_file_inventory.csv`. The preview below shows parsed identifiers, detected modality, readability, and safe CSV preview metadata.

In [3]:
preview_cols = [
    "relative_path", "extension", "subject_id", "session_id", "difficulty_level", "run_id",
    "detected_modality", "modality_detection_method", "is_readable",
    "row_count_if_csv", "column_count_if_csv", "first_columns_if_csv",
]
display(inventory[preview_cols].head(12))

,relative_path,extension,subject_id,session_id,difficulty_level,run_id,detected_modality,modality_detection_method,is_readable,row_count_if_csv,column_count_if_csv,first_columns_if_csv
0,task-ils/PerfMetrics.csv,csv,,,,,performance,filename_keyword,True,419,8,"[""subject"", ""date"", ""run"", ""difficulty"", ""cumu..."
1,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,eye_tracking,filename_keyword,True,1,27,"[""time_dn"", ""validity_l"", ""validity_r"", ""gaze_..."
2,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,eye_tracking,filename_keyword,True,1,4,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
3,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,ecg,filename_keyword,True,269979,4,"[""time_dn"", ""ecg_projection_ll_ra_mV"", ""ecg_pr..."
4,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,ecg,filename_keyword,True,1,4,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
5,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,eda,filename_keyword,True,68412,3,"[""time_dn"", ""ppg_finger_mV"", ""eda_hand_l_kOhms""]"
6,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,eda,filename_keyword,True,1,4,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
7,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,emg,filename_keyword,True,271951,6,"[""time_dn"", ""accelerometry_forearm_r_x_mps2"", ..."
8,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,emg,filename_keyword,True,1,4,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
9,task-ils/sub-cp003/ses-20210206/level-01B_run-...,csv,sub-cp003,ses-20210206,level-01B,run-001,respiration,filename_keyword,True,269979,2,"[""time_dn"", ""respiration_trace_mV""]"


## Difficulty Level Distribution

This table counts parsed run keys by difficulty level. Rest-task runs appear as `level-000`; ILS task difficulty levels appear as the workload levels used for later modeling.

In [4]:
display(summaries["difficulty_distribution"])
print("Difficulty distribution figure saved to figures/difficulty_distribution.png")

,difficulty_level,run_count
0,level-000,68
1,level-01B,104
2,level-02B,106
3,level-03B,104
4,level-04B,105


Difficulty distribution figure saved to figures/difficulty_distribution.png


## Modality Availability Summary

The run-level table records whether each parsed run has eye tracking, ECG, EDA, EMG, respiration, head movement, X-Plane, control input, and performance data.

In [5]:
display(summaries["modality_summary"])
display(summaries["modality_availability_summary"])
print("Modality availability figure saved to figures/modality_availability_bar.png")

,detected_modality,file_count
8,xplane,1461
3,eye_tracking,1443
6,respiration,1386
0,ecg,974
2,emg,974
1,eda,950
7,unknown,908
4,head_movement,487
5,performance,420


,modality,run_count,missing_run_count,run_coverage_percent
0,eye_tracking,487,0,100.00
1,ecg,487,0,100.00
2,eda,475,12,97.54
3,emg,487,0,100.00
4,respiration,487,0,100.00
5,head_movement,487,0,100.00
6,xplane,487,0,100.00
7,control_input,0,487,0.00
8,performance,419,68,86.04


Modality availability figure saved to figures/modality_availability_bar.png


## Run-Level Modality Availability Preview

The complete run-level modality availability table is saved to `results/run_modality_availability.csv`.

In [6]:
display(run_availability.head(12))

,subject_id,session_id,difficulty_level,run_id,run_key,has_eye_tracking,has_ecg,has_eda,has_emg,has_respiration,has_head_movement,has_xplane,has_control_input,has_performance,available_modality_count,available_modalities
0,sub-cp003,ses-20210206,level-000,run-001,sub-cp003_ses-20210206_level-000_run-001,True,True,True,True,True,True,True,False,False,7,ecg;eda;emg;eye_tracking;head_movement;respira...
1,sub-cp003,ses-20210206,level-000,run-002,sub-cp003_ses-20210206_level-000_run-002,True,True,True,True,True,True,True,False,False,7,ecg;eda;emg;eye_tracking;head_movement;respira...
2,sub-cp003,ses-20210206,level-01B,run-001,sub-cp003_ses-20210206_level-01B_run-001,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
3,sub-cp003,ses-20210206,level-01B,run-007,sub-cp003_ses-20210206_level-01B_run-007,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
4,sub-cp003,ses-20210206,level-01B,run-012,sub-cp003_ses-20210206_level-01B_run-012,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
5,sub-cp003,ses-20210206,level-02B,run-003,sub-cp003_ses-20210206_level-02B_run-003,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
6,sub-cp003,ses-20210206,level-02B,run-008,sub-cp003_ses-20210206_level-02B_run-008,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
7,sub-cp003,ses-20210206,level-02B,run-010,sub-cp003_ses-20210206_level-02B_run-010,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
8,sub-cp003,ses-20210206,level-03B,run-002,sub-cp003_ses-20210206_level-03B_run-002,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
9,sub-cp003,ses-20210206,level-03B,run-005,sub-cp003_ses-20210206_level-03B_run-005,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...


## Missing, Unknown, and Unreadable Files

Unknown files are intentionally not guessed. They are saved for manual review. Unreadable files, if any, are also saved separately.

In [7]:
missing_summary = summaries["modality_availability_summary"][["modality", "missing_run_count", "run_coverage_percent"]]
display(missing_summary)

print(f"Unknown files requiring manual review: {report['unknown_file_count']}")
if report['unknown_file_count']:
    display(summaries["unknown_files"][["relative_path", "file_name", "modality_detection_method", "first_columns_if_csv"]].head(20))

print(f"Unreadable files: {report['unreadable_file_count']}")
if report['unreadable_file_count']:
    display(summaries["unreadable_files"][["relative_path", "file_name", "read_error"]].head(20))

print(f"Metadata/license files not modeling data: {report['metadata_or_license_file_count']}")

,modality,missing_run_count,run_coverage_percent
0,eye_tracking,0,100.00
1,ecg,0,100.00
2,eda,12,97.54
3,emg,0,100.00
4,respiration,0,100.00
5,head_movement,0,100.00
6,xplane,0,100.00
7,control_input,487,0.00
8,performance,68,86.04


Unknown files requiring manual review: 908


,relative_path,file_name,modality_detection_method,first_columns_if_csv
192,task-ils/sub-cp004/ses-20210330/level-01B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""time_dn"", ""accelerometry_torso_x_mps2"", ""acc..."
193,task-ils/sub-cp004/ses-20210330/level-01B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
210,task-ils/sub-cp004/ses-20210330/level-01B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""time_dn"", ""accelerometry_torso_x_mps2"", ""acc..."
211,task-ils/sub-cp004/ses-20210330/level-01B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
228,task-ils/sub-cp004/ses-20210330/level-01B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""time_dn"", ""accelerometry_torso_x_mps2"", ""acc..."
229,task-ils/sub-cp004/ses-20210330/level-01B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
246,task-ils/sub-cp004/ses-20210330/level-02B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""time_dn"", ""accelerometry_torso_x_mps2"", ""acc..."
247,task-ils/sub-cp004/ses-20210330/level-02B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."
264,task-ils/sub-cp004/ses-20210330/level-02B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""time_dn"", ""accelerometry_torso_x_mps2"", ""acc..."
265,task-ils/sub-cp004/ses-20210330/level-02B_run-...,sub-cp004_ses-20210330_task-ils_stream-lslshim...,unclassified_filename,"[""processedTime"", ""Fs_Hz"", ""Fs_Hz_effective"", ..."


Unreadable files: 0
Metadata/license files not modeling data: 0


## Complete Multimodal Runs and Phase 02 Feasibility

Strict complete runs require every requested modality, including control input. A second core definition excludes control input because no reliable control-input modality should be inferred unless manually confirmed.

In [8]:
display(summaries["complete_runs_summary"])
print(f"Strict full multimodal feature extraction feasible: {report['full_multimodal_feature_extraction_feasible']}")
print(f"Core multimodal feature extraction feasible without control input: {report['core_multimodal_feature_extraction_feasible_without_control_input']}")
print(f"Manual modality mapping required before Phase 02: {report['manual_mapping_required_before_phase_02']}")

,definition,required_modalities,complete_run_count,total_run_count,coverage_percent
0,strict_all_listed_modalities,eye_tracking;ecg;eda;emg;respiration;head_move...,0,487,0.00
1,core_multimodal_without_control_input,eye_tracking;ecg;eda;emg;respiration;head_move...,408,487,83.78


Strict full multimodal feature extraction feasible: False
Core multimodal feature extraction feasible without control input: True
Manual modality mapping required before Phase 02: True


## Saved Outputs

The following cell verifies that the requested Phase 01 artifacts were written under the experiment directory.

In [9]:
expected_outputs = [
    "results/raw_file_inventory.csv",
    "results/run_modality_availability.csv",
    "results/modality_summary.csv",
    "results/unknown_files_for_review.csv",
    "results/unreadable_files.csv",
    "results/raw_data_audit_report.json",
    "tables/difficulty_distribution.csv",
    "tables/modality_availability_summary.csv",
    "tables/complete_runs_summary.csv",
    "figures/modality_availability_bar.png",
    "figures/difficulty_distribution.png",
    "logs/phase_01_log.txt",
    "README.md",
]
for rel in expected_outputs:
    path = PHASE_DIR / rel
    print(f"{rel}: {'OK' if path.exists() else 'MISSING'}")

results/raw_file_inventory.csv: OK
results/run_modality_availability.csv: OK
results/modality_summary.csv: OK
results/unknown_files_for_review.csv: OK
results/unreadable_files.csv: OK
results/raw_data_audit_report.json: OK
tables/difficulty_distribution.csv: OK
tables/modality_availability_summary.csv: OK
tables/complete_runs_summary.csv: OK
figures/modality_availability_bar.png: OK
figures/difficulty_distribution.png: OK
logs/phase_01_log.txt: OK
README.md: OK
